# Thrust Chamber Assembly (TCA) Sizing
### LOX / Isopropanol (IPA) Bipropellant Engine
---
Uses NASA CEA via `rocketcea` to size the thrust chamber geometry from user-defined parameters.

**Outputs:** C\*, Cf, Isp, mass flow rates, throat/exit/chamber dimensions, γ, molecular weight.

## 1. Imports

In [182]:
import numpy as np
import csv
import yaml
from rocketcea.cea_obj import CEA_Obj

np.set_printoptions(legacy='1.25')  # Fix for numpy float printing issues

## 2. Unit Conversion Constants

In [183]:
# ── Unit conversions ──────────────────────────────────────────────────────────
PSI_TO_PA       = 6894.76        # psi  → Pa
FT_TO_M         = 0.3048         # ft   → m
M_TO_CM         = 100.0          # m    → cm
M_TO_IN         = 39.3701        # m    → in
RANKINE_TO_K    = 5.0 / 9.0      # °R   → K
RANKINE_TO_F    = -459.67        # °R   → °F  (offset, not factor)
LBF_TO_N        = 4.4482216      # lbf  → N
KG_TO_LBM       = 2.20462        # kg   → lbm
Ru              = 8314.462618    # Universal gas constant  [J / (kmol·K)]
g0              = 9.80665        # Standard gravity        [m/s²]

## 3. Load Parameters from YAML

Edit `TCA_params.yaml` to change engine inputs. Expected keys:

| Key | Description | Units |
|-----|-------------|-------|
| `thrust` | Target thrust | lbf |
| `tca_chamber_pressure` | Chamber pressure | psia |
| `tca_exit_pressure` | Nozzle exit pressure | psia |
| `ambient_pressure` | Ambient pressure | psia |
| `oxidizer_fuel_ratio` | O/F mass ratio | — |
| `c_star_efficiency` | C\* efficiency (η_c\*) | — |
| `thrust_coefficient_efficiency` | Cf efficiency (η_Cf) | — |
| `characteristic_length` | L\* | inches |
| `tca_convergent_half_angle` | Convergent half-angle | degrees |
| `tca_chamber_diameter` | Target chamber diameter | inches |

In [ ]:
with open('../TCA_params.yaml') as f:
    p = yaml.safe_load(f)

# ── Engine requirements ───────────────────────────────────────────────────────
F_lbf       = p['thrust']                      # Target thrust              [lbf]
Pc_psia     = p['tca_chamber_pressure']         # Chamber pressure           [psia]
Pe_psia     = p['tca_exit_pressure']            # Exit pressure              [psia]
Pamb_psia   = p['ambient_pressure']             # Ambient pressure           [psia]
OF          = p['oxidizer_fuel_ratio']          # O/F ratio                  [—]

# ── Efficiency factors ────────────────────────────────────────────────────────
eta_cstar   = p['c_star_efficiency']            # C* efficiency              [—]
eta_cf      = p['thrust_coefficient_efficiency']# Cf efficiency              [—]

# ── Chamber geometry inputs ───────────────────────────────────────────────────
L_star_in   = p['characteristic_length']        # Characteristic length      [in]
alpha_deg   = p['tca_convergent_half_angle']    # Convergent half-angle      [deg]
Dc_input_in = p['tca_chamber_diameter']         # Target chamber diameter    [in]

# ── Derived unit conversions ──────────────────────────────────────────────────
F_N         = F_lbf * LBF_TO_N                 # Thrust                     [N]
Pc_Pa       = Pc_psia * PSI_TO_PA               # Chamber pressure           [Pa]
L_star_cm   = L_star_in / 12 * FT_TO_M * M_TO_CM  # L*                     [cm]

print(f"Loaded parameters: F={F_lbf} lbf, Pc={Pc_psia} psia, O/F={OF}, η_c*={eta_cstar}, η_Cf={eta_cf}")

Loaded parameters: F=5000 lbf, Pc=500 psia, O/F=1.6, η_c*=0.875, η_Cf=0.875


## 4. CEA Calculations

In [185]:
# ── Initialise CEA object ─────────────────────────────────────────────────────
cea = CEA_Obj(oxName='LOX', fuelName='Isopropanol')

# ── Optimal expansion ratio at given Pc/Pe ────────────────────────────────────
eps = cea.get_eps_at_PcOvPe(Pc=Pc_psia, MR=OF, PcOvPe=(Pc_psia / Pe_psia))

# ── Characteristic velocity C* ────────────────────────────────────────────────
Cstar_ideal_ms  = cea.get_Cstar(Pc=Pc_psia, MR=OF) * FT_TO_M   # ideal C* [m/s]
Cstar_ms        = Cstar_ideal_ms * eta_cstar                     # real  C* [m/s]

# ── Thrust coefficient Cf ─────────────────────────────────────────────────────
cf_arr          = cea.get_PambCf(Pamb=Pamb_psia, Pc=Pc_psia, MR=OF, eps=eps)
Cf_ideal        = cf_arr[0]                                      # ideal Cf [—]
Cf              = Cf_ideal * eta_cf                              # real  Cf [—]

# ── Specific impulse ──────────────────────────────────────────────────────────
Isp_s           = Cf * Cstar_ms / g0                             # Isp      [s]

# ── Combustion temperatures ───────────────────────────────────────────────────
Tc_R, Tt_R, Te_R = cea.get_Temperatures(Pc=Pc_psia, MR=OF, eps=eps,
                                         frozen=0, frozenAtThroat=0)
Tc_K = Tc_R * RANKINE_TO_K
Tt_K = Tt_R * RANKINE_TO_K
Te_K = Te_R * RANKINE_TO_K

# ── Thermodynamic properties (chamber) — get_Chamber_MolWt_gamma is the
#    correct source for γ used in performance calculations ────────────────────
MW_chamber, gamma_chamber = cea.get_Chamber_MolWt_gamma(Pc=Pc_psia, MR=OF, eps=eps)
R_gas = Ru / MW_chamber                                          # gas constant [J/(kg·K)]

# ── Thermodynamic properties (throat) ────────────────────────────────────────
MW_throat, gamma_throat = cea.get_Throat_MolWt_gamma(Pc=Pc_psia, MR=OF, eps=eps, frozen=0)

# ── Transport properties (separate from γ — do NOT mix up) ───────────────────
gamma_transport, viscosity, k_conductivity, Prandtl = cea.get_Chamber_Transport(
    Pc=Pc_psia, MR=OF, eps=eps, frozen=0
)
# NOTE: gamma_transport ≠ gamma_chamber; transport gamma is used only for
# heat-transfer correlations, NOT for nozzle / Isp calculations.

print("CEA calculations complete.")

CEA calculations complete.


## 5. Mass Flow Rates

In [186]:
# ── Total mass flow rate ──────────────────────────────────────────────────────
# Derived from F = Cf · C* · mdot  →  mdot = F / (Cf · C*)  [kg/s]
mdot_kgs    = F_N / (Cf * Cstar_ms)             # total mass flow rate  [kg/s]
mdot_lbms   = mdot_kgs * KG_TO_LBM              # total mass flow rate  [lbm/s]

# ── Propellant split ──────────────────────────────────────────────────────────
mdot_ipa_lbms = mdot_lbms / (1 + OF)            # IPA flow rate         [lbm/s]
mdot_lox_lbms = mdot_lbms * OF / (1 + OF)       # LOX flow rate         [lbm/s]

print(f"Total ṁ = {mdot_lbms:.4f} lbm/s  ({mdot_kgs:.4f} kg/s)")
print(f"  IPA  ṁ = {mdot_ipa_lbms:.4f} lbm/s")
print(f"  LOX  ṁ = {mdot_lox_lbms:.4f} lbm/s")

Total ṁ = 23.6300 lbm/s  (10.7184 kg/s)
  IPA  ṁ = 9.0885 lbm/s
  LOX  ṁ = 14.5416 lbm/s


## 6. Throat & Exit Geometry

In [187]:
# ── Throat area ───────────────────────────────────────────────────────────────
# From continuity: mdot = (At · Pc) / C*
At_m2   = (mdot_kgs * Cstar_ms) / Pc_Pa         # throat area  [m²]
At_in2  = At_m2 * (M_TO_IN ** 2)                # throat area  [in²]
At_cm2  = At_m2 * (M_TO_CM ** 2)                # throat area  [cm²]
Dt_in   = 2 * np.sqrt(At_in2 / np.pi)           # throat dia.  [in]
Rt_cm   = np.sqrt(At_cm2 / np.pi)               # throat rad.  [cm]

# ── Exit area ─────────────────────────────────────────────────────────────────
Ae_in2  = At_in2 * eps                           # exit area    [in²]
De_in   = 2 * np.sqrt(Ae_in2 / np.pi)           # exit dia.    [in]

print(f"Throat:  Dt = {Dt_in:.4f} in  |  At = {At_in2:.4f} in²")
print(f"Exit:    De = {De_in:.4f} in  |  Ae = {Ae_in2:.4f} in²")
print(f"Expansion ratio ε = {eps:.4f}")

Throat:  Dt = 3.0746 in  |  At = 7.4247 in²
Exit:    De = 7.7328 in  |  Ae = 46.9634 in²
Expansion ratio ε = 6.3253


## 7. Chamber Geometry

In [188]:
# ── Contraction ratio from input chamber diameter ─────────────────────────────
# Area ratio = (Dc/Dt)²  for circular cross-sections
con_r   = (Dc_input_in / Dt_in) ** 2            # contraction ratio    [—]
Ac_in2  = At_in2 * con_r                        # chamber area         [in²]
Dc_in   = 2 * np.sqrt(Ac_in2 / np.pi)          # chamber dia. (recalc)[in]

# ── Chamber cylindrical length from L* ───────────────────────────────────────
# L* = Vc / At  where Vc includes the cylindrical section + converging cone
# Lc = (L*_cm - V_cone/At_cm) / con_r
# V_cone/At_cm = (1/3) * Rt_cm * (1/tan(α)) * (con_r^(1/3) - 1)
alpha_rad = np.deg2rad(alpha_deg)

Rt_cm = np.sqrt(At_cm2 / np.pi)              # throat radius   [cm]
Rc_cm = Rt_cm * np.sqrt(con_r)               # chamber radius  [cm]

# Frustum (truncated cone) volume [cm³]
L_cone_cm   = (Rc_cm - Rt_cm) / np.tan(alpha_rad)
V_cone_cm3  = (np.pi / 3) * (Rc_cm**2 + Rc_cm * Rt_cm + Rt_cm**2) * L_cone_cm

# Cylindrical section volume [cm³]
Vc_total_cm3 = L_star_cm * At_cm2            # total required volume
V_cyl_cm3    = Vc_total_cm3 - V_cone_cm3     # subtract cone

# Cylindrical length
Lc_cm  = V_cyl_cm3 / (np.pi * Rc_cm**2)
Lc_in  = Lc_cm / M_TO_CM / FT_TO_M * 12

# Total length injector → throat
L_cone_in = L_cone_cm / M_TO_CM / FT_TO_M * 12
L_total_in = Lc_in + L_cone_in

print(f"Chamber: Dc = {Dc_in:.4f} in")
print(f"Contraction ratio = {con_r:.3f}")
print(f"Cylindrical chamber length Lc = {Lc_in:.4f} in")

Chamber: Dc = 5.1890 in
Contraction ratio = 2.848
Cylindrical chamber length Lc = 13.3587 in


## 8. Residence Time

In [189]:
# t_s = L* / C*   [s]
L_star_m    = L_star_in / M_TO_IN               # L*  [m]
t_stay_ms   = (L_star_m / Cstar_ms) * 1000      # residence time [ms]

print(f"L* = {L_star_in:.2f} in  =  {L_star_m*100:.2f} cm")
print(f"Residence time t_s = L*/C* = {t_stay_ms:.3f} ms")

L* = 40.00 in  =  101.60 cm
Residence time t_s = L*/C* = 0.659 ms


## 9. Summary Print

In [190]:
print("=" * 55)
print("  TCA SIZING RESULTS  —  LOX / IPA")
print("=" * 55)

print("\n── Performance ──────────────────────────────────")
print(f"  C* (ideal)          : {Cstar_ideal_ms:>10.3f}  m/s")
print(f"  C* (real, η={eta_cstar}) : {Cstar_ms:>10.3f}  m/s")
print(f"  Cf (ideal)          : {Cf_ideal:>10.4f}")
print(f"  Cf (real, η={eta_cf})  : {Cf:>10.4f}")
print(f"  Isp                 : {Isp_s:>10.3f}  s")
print(f"  Thrust (calc)       : {At_in2 * Pc_psia * Cf:>10.3f}  lbf")

print("\n── Combustion ───────────────────────────────────")
print(f"  Tc                  : {Tc_K:>10.1f}  K")
print(f"  Tt                  : {Tt_K:>10.1f}  K")
print(f"  Te                  : {Te_K:>10.1f}  K")
print(f"  Mol. weight (chamber): {MW_chamber:>9.3f}  lbm/lbmol")
print(f"  γ (chamber, thermo) : {gamma_chamber:>10.4f}")
print(f"  γ (throat, thermo)  : {gamma_throat:>10.4f}")
print(f"  γ (transport only)  : {gamma_transport:>10.4f}")
print(f"  Viscosity           : {viscosity:>10.4f}  mPa·s")
print(f"  Thermal conductivity: {k_conductivity:>10.4f}  mcal/(cm·K·s)")
print(f"  Prandtl number      : {Prandtl:>10.4f}")

print("\n── Mass Flow Rates ──────────────────────────────")
print(f"  ṁ total             : {mdot_lbms:>10.4f}  lbm/s")
print(f"  ṁ IPA               : {mdot_ipa_lbms:>10.4f}  lbm/s")
print(f"  ṁ LOX               : {mdot_lox_lbms:>10.4f}  lbm/s")

print("\n── Geometry ─────────────────────────────────────")
print(f"  Throat diameter Dt  : {Dt_in:>10.4f}  in")
print(f"  Throat area At      : {At_in2:>10.4f}  in²")
print(f"  Exit diameter De    : {De_in:>10.4f}  in")
print(f"  Exit area Ae        : {Ae_in2:>10.4f}  in²")
print(f"  Expansion ratio ε   : {eps:>10.4f}")
print(f"  Chamber diameter Dc : {Dc_in:>10.4f}  in")
print(f"  Contraction ratio   : {con_r:>10.3f}")
print(f"  Chamber length Lc   : {Lc_in:>10.4f}  in")
print(f"  L*                  : {L_star_in:>10.2f}  in")
print(f"  Residence time t_s  : {t_stay_ms:>10.3f}  ms")
print("=" * 55)

  TCA SIZING RESULTS  —  LOX / IPA

── Performance ──────────────────────────────────
  C* (ideal)          :   1760.737  m/s
  C* (real, η=0.875) :   1540.644  m/s
  Cf (ideal)          :     1.5393
  Cf (real, η=0.875)  :     1.3469
  Isp                 :    211.595  s
  Thrust (calc)       :   5000.003  lbf

── Combustion ───────────────────────────────────
  Tc                  :     3284.5  K
  Tt                  :     3088.0  K
  Te                  :     1944.2  K
  Mol. weight (chamber):    21.625  lbm/lbmol
  γ (chamber, thermo) :     1.1461
  γ (throat, thermo)  :     1.1500
  γ (transport only)  :     1.1323
  Viscosity           :     1.0345  mPa·s
  Thermal conductivity:     2.4321  mcal/(cm·K·s)
  Prandtl number      :     0.4816

── Mass Flow Rates ──────────────────────────────
  ṁ total             :    23.6300  lbm/s
  ṁ IPA               :     9.0885  lbm/s
  ṁ LOX               :    14.5416  lbm/s

── Geometry ─────────────────────────────────────
  Throat diamete

## 10. Export to CSV

In [191]:
output_file_path = 'Dimensions_CSV/cea_dimensions.csv'

headers = [
    'Thrust (lbf)', 'Chamber Pressure (psia)', 'Exit Pressure (psia)',
    'O/F Ratio', 'η C*', 'η Cf',
    'L* (in)', 'Convergent Half-Angle (deg)', 'Contraction Ratio',
    'Throat Diameter (in)', 'Throat Area (in²)',
    'Exit Diameter (in)',   'Exit Area (in²)',
    'Expansion Ratio',
    'Chamber Diameter (in)', 'Chamber Length (in)',
    'C* real (m/s)', 'Cf real', 'Isp (s)',
    'Tc (K)', 'Tt (K)', 'Te (K)',
    'γ chamber', 'MW chamber (lbm/lbmol)',
    'ṁ total (lbm/s)', 'ṁ IPA (lbm/s)', 'ṁ LOX (lbm/s)',
    'Residence time (ms)'
]

values = [
    round(F_lbf, 3),        round(Pc_psia, 3),      round(Pe_psia, 3),
    round(OF, 3),           round(eta_cstar, 3),    round(eta_cf, 3),
    round(L_star_in, 2),    round(alpha_deg, 1),    round(con_r, 3),
    round(Dt_in, 4),        round(At_in2, 4),
    round(De_in, 4),        round(Ae_in2, 4),
    round(eps, 4),
    round(Dc_in, 4),        round(Lc_in, 4),
    round(Cstar_ms, 3),     round(Cf, 4),           round(Isp_s, 3),
    round(Tc_K, 1),         round(Tt_K, 1),         round(Te_K, 1),
    round(gamma_chamber, 4),round(MW_chamber, 3),
    round(mdot_lbms, 4),    round(mdot_ipa_lbms, 4),round(mdot_lox_lbms, 4),
    round(t_stay_ms, 4)
]

with open(output_file_path, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(headers)
    writer.writerow(values)

print(f"Results exported to: {output_file_path}")

Results exported to: Dimensions_CSV/cea_dimensions.csv
